In [34]:
import time
from enum import Enum
from dataclasses import dataclass

import numpy as np
from termcolor import colored, cprint

from utils import Converts

from XPlaneConnectX import XPlaneConnectX
from ics_connector import ICSInputs, ICSOutputs, ICSBenchConnector

In [35]:
RWY_START_LAT = 55.977874756
RWY_START_LON = 37.440910339
RWY_END_LAT = 55.969951630
RWY_END_LON = 37.387878418
RWY_HEADING_TRUE = 255.45  # Истинный курс полосы для инициализации
ELEVATION_MSL = 188.976  # Высота порога над уровнем моря (620 фута в метрах)
ELEVATION_AIRCRAFT = 3.0 * Converts.FT_TO_M # Высота ЛА над ВПП

INITIAL_SPEED_KTS = 160.0
TARGET_SPEED_KTS = 10.0

A330_SETUP = dict(
    speed_knots=INITIAL_SPEED_KTS,
    descent_rate_fpm=130.0,
    pitch_deg=-2.0
)
xpc = XPlaneConnectX(ip="127.0.0.1", port=49000)

In [36]:
def setup_touchdown_uuee_24c(xpc: XPlaneConnectX, speed_knots: float = 140.0, descent_rate_fpm: float = 120.0,
                             pitch_deg: float = 0.0):
    """
    Мгновенно воссоздает этап касания (touchdown) самолета на ВПП 24C аэропорта Шереметьево (UUEE)
    с последующим корректным физическим расчетом пробега, обжатия шасси и торможения.

    :param xpc: Экземпляр инициализированного класса XPlaneConnectX
    :param speed_knots: Путевая скорость самолета в узлах в момент касания
    :param descent_rate_fpm: Вертикальная скорость снижения в футах в минуту (положительное число)
    :param pitch_deg: Угол тангажа (типичный угол кабрирования при касании основных стоек)
    """
    # 1. Замораживаем физическое время симулятора.
    # Это необходимо, чтобы все сетевые пакеты (позиция, конфигурация, скорости)
    # применились в рамках одного расчетного кадра до начала обсчета физики.
    xpc.pauseSIM(True)
    time.sleep(0.1)  # Короткая пауза для стабилизации сетевого цикла X-Plane

    roll_deg = 0.0  # Крен равен нулю (крылья параллельны ВПП)

    # 3. Установка пространственного положения планера
    xpc.sendPOSI(lat=RWY_START_LAT, lon=RWY_START_LON, elev=ELEVATION_MSL + ELEVATION_AIRCRAFT, phi=roll_deg,
                 theta=pitch_deg,
                 psi_true=RWY_HEADING_TRUE)

    # 4. Конфигурация органов управления и механизации крыла
    # Тяга на малый газ (0.0), шасси выпущено (1), закрылки на максимум (1.0), спидбрейки армированы (-0.5)
    xpc.sendCTRL(lat_control=0.0, lon_control=0.0, rudder_control=0.0,
                 throttle=0.0, gear=1, flaps=1.0, speedbrakes=-0.5, park_brake=0.0)

    # 5. Расчет проекций векторов скоростей в локальной системе координат X-Plane (OpenGL)
    # Перевод исходных параметров в систему СИ (метры в секунду)
    v_ground_ms = speed_knots * Converts.KTS_TO_MS
    v_vertical_ms = -abs(descent_rate_fpm) * Converts.FTM_TO_MS  # Гарантируем отрицательное значение (движение вниз)

    # Перевод истинного курса в радианы
    heading_rad = np.radians(RWY_HEADING_TRUE)

    # Математический расчет проекций скоростей на оси декартовой сетки X-Plane:
    # Ось X направлена на Восток, ось Z — на Юг.
    local_vx = v_ground_ms * np.sin(heading_rad)
    local_vz = -v_ground_ms * np.cos(heading_rad)
    local_vy = v_vertical_ms

    # 6. Запись векторов скоростей напрямую в DataRefs через команду DREF
    xpc.sendDREF("sim/flightmodel/position/local_vx", local_vx)
    xpc.sendDREF("sim/flightmodel/position/local_vy", local_vy)
    xpc.sendDREF("sim/flightmodel/position/local_vz", local_vz)

    # Демпфирование (обнуление) угловых скоростей для предотвращения паразитного вращения планера
    xpc.sendDREF("sim/flightmodel/position/P", 0.0)
    xpc.sendDREF("sim/flightmodel/position/Q", 0.0)
    xpc.sendDREF("sim/flightmodel/position/R", 0.0)

    time.sleep(0.1)
    # 7. Возобновление симуляции. Физический движок Blade Element Theory мгновенно подхватывает
    # заданные параметры движения и рассчитывает удар о полосу, обжатие амортизаторов и пробег.
    xpc.pauseSIM(False)

In [37]:
class FailureMode(Enum):
    NONE = 0

    GEAR_CONFIG = 1

    STABILIZER_FAIL = 2

    ENGINE_OUT_LEFT = 3
    ENGINE_OUT_RIGHT = 4

    NWS_FAIL = 5

    REVERSE_LEFT_FAIL = 6
    REVERSE_RIGHT_FAIL = 7

    THRUST_LEFT_DEGRADED = 8
    THRUST_RIGHT_DEGRADED = 9


@dataclass
class FailureState:
    steering_eff = 1.0

    brake_left_eff = 1.0
    brake_right_eff = 1.0

    reverse_left_eff = 1.0
    reverse_right_eff = 1.0

    thrust_left_eff = 1.0
    thrust_right_eff = 1.0

    elevator_eff = 1.0

    gear_conflict = False


class FailureManager:
    def __init__(self):
        self.state = FailureState()

    def reset(self):
        self.state = FailureState()

    def activate(self, mode):
        match mode:
            case FailureMode.NWS_FAIL:
                self.state.steering_eff = 0.0

            case FailureMode.ENGINE_OUT_LEFT:
                self.state.thrust_left_eff = 0.0

            case FailureMode.ENGINE_OUT_RIGHT:
                self.state.thrust_right_eff = 0.0

            case FailureMode.REVERSE_LEFT_FAIL:
                self.state.reverse_left_eff = 0.0

            case FailureMode.REVERSE_RIGHT_FAIL:
                self.state.reverse_right_eff = 0.0

            case FailureMode.STABILIZER_FAIL:
                self.state.elevator_eff = 0.0

            case FailureMode.GEAR_CONFIG:
                self.state.gear_conflict = True
                self.state.brake_left_eff = 0.6
                self.state.brake_right_eff = 0.6

In [38]:
# class XPlaneFailureManager:
#     def __init__(self, xpc_client: XPlaneConnectX):
#         self.xpc = xpc_client
#
#         # Значения перечислений (enums) для X-Plane отказов
#         self.FAIL_OK = 0       # Система исправна
#         self.FAIL_NOW = 6      # Немедленный жесткий отказ
#
#     def _set_failure(self, dref_path: str, value: int):
#         """Вспомогательный метод отправки команды отказа в симулятор"""
#         try:
#             self.xpc.sendDREF(dref_path, value)
#         except Exception as e:
#             print(f"Ошибка при отправке DataRef {dref_path}: {e}")
#
#     # --- 1. ШАССИ И ТОРМОЗА ---
#     def set_gear_actuator_failure(self, state: bool):
#         """Отказ приводов выпуска/уборки шасси (клинят в текущем положении)"""
#         val = self.FAIL_NOW if state else self.FAIL_OK
#         print(f"[FAIL MGR] Отказ гидросистемы/приводов шасси -> {state}")
#         self._set_failure("sim/operation/failures/rel_gear_act", val)
#
#     def set_main_gear_asymmetry(self, left_failed: bool, right_failed: bool):
#         """Отказ конкретных стоек шасси (выпуск/замки)"""
#         val_l = self.FAIL_NOW if left_failed else self.FAIL_OK
#         val_r = self.FAIL_NOW if right_failed else self.FAIL_OK
#         print(f"[FAIL MGR] Индивидуальный отказ стоек: Левая={left_failed}, Правая={right_failed}")
#         self._set_failure("sim/operation/failures/rel_Lgear", val_l)
#         self._set_failure("sim/operation/failures/rel_Rgear", val_r)
#
#     # --- 2. СТАБИЛИЗАТОР ---
#     def set_stabilizer_jam(self, state: bool):
#         """Заклинивание переставного стабилизатора (Pitch Trim)"""
#         val = self.FAIL_NOW if state else self.FAIL_OK
#         print(f"[FAIL MGR] Заклинивание триммера стабилизатора -> {state}")
#         self._set_failure("sim/operation/failures/rel_trim_pitch", val)
#
#     # --- 3. ДВИГАТЕЛИ И ТЯГА ---
#     def set_engine_flameout(self, engine_index: int, state: bool):
#         """Полный отказ двигателя (Flameout). engine_index: 0 = Левый, 1 = Правый"""
#         val = self.FAIL_NOW if state else self.FAIL_OK
#         dref = f"sim/operation/failures/rel_engfai{engine_index}"
#         print(f"[FAIL MGR] Полный отказ двигателя [{engine_index}] -> {state}")
#         self._set_failure(dref, val)
#
#     def set_engine_compressor_stall(self, engine_index: int, state: bool):
#         """Помпаж компрессора (нарушение стабильности тяги)"""
#         val = self.FAIL_NOW if state else self.FAIL_OK
#         dref = f"sim/operation/failures/rel_cop{engine_index}"
#         print(f"[FAIL MGR] Помпаж компрессора двигателя [{engine_index}] -> {state}")
#         self._set_failure(dref, val)
#
#     # --- 4. НОСОВАЯ СТОЙКА (УПРАВЛЕНИЕ) ---
#     def set_nosewheel_steering_failure(self, state: bool):
#         """Отказ рулевого управления носовой стойки (переходит в свободное ориентирование)"""
#         val = self.FAIL_NOW if state else self.FAIL_OK
#         print(f"[FAIL MGR] Отказ управления носовой стойки (свободный кастор) -> {state}")
#         self._set_failure("sim/operation/failures/rel_steer_nose", val)
#
#     # --- 5. РЕВЕРС ТЯГИ ---
#     def set_reverse_thrust_failure(self, engine_index: int, state: bool):
#         """Отказ привода реверса. engine_index: 0 = Левый, 1 = Правый"""
#         val = self.FAIL_NOW if state else self.FAIL_OK
#         dref = f"sim/operation/failures/rel_revfail{engine_index}"
#         print(f"[FAIL MGR] Отказ механизма реверса двигателя [{engine_index}] -> {state}")
#         self._set_failure(dref, val)
#
#     def clear_all_failures(self):
#         """Полный сброс всех критических отказов в системе управления пробегом"""
#         print("[FAIL MGR] Восстановление всех систем в штатный режим...")
#         self.set_gear_actuator_failure(False)
#         self.set_main_gear_asymmetry(False, False)
#         self.set_stabilizer_jam(False)
#         self.set_engine_flameout(0, False)
#         self.set_engine_flameout(1, False)
#         self.set_engine_compressor_stall(0, False)
#         self.set_engine_compressor_stall(1, False)
#         self.set_nosewheel_steering_failure(False)
#         self.set_reverse_thrust_failure(0, False)
#         self.set_reverse_thrust_failure(1, False)
#
#
# # --- Демонстрационный сценарий тестирования ---
# if __name__ == "__main__":
#     # Инициализируем клиент подключения к X-Plane
#     xpc = XPlaneConnectX(ip="127.0.0.1", port=49000)
#     fail_mgr = XPlaneFailureManager(xpc)
#
#     # Перед началом теста убеждаемся, что самолет чист от отказов
#     fail_mgr.clear_all_failures()
#     time.sleep(2)
#
#     print("\n--- Старт имитационного сценария пробега ЛА ---")
#
#     # Имитируем: коснулись полосы, скорость 130 узлов. Включаем реверс.
#     # Через 3 секунды ломается реверс правого двигателя (№1)
#     time.sleep(3)
#     fail_mgr.set_reverse_thrust_failure(engine_index=1, state=True)
#
#     # Скорость падает до 90 узлов, пилот пытается удержать ось носовой стойкой.
#     # Происходит отказ управления носовой стойки.
#     time.sleep(4)
#     fail_mgr.set_nosewheel_steering_failure(state=True)
#
#     # На скорости 50 узлов лопается гидролиния левой стойки шасси
#     time.sleep(4)
#     fail_mgr.set_main_gear_asymmetry(left_failed=True, right_failed=False)
#
#     # Конец теста, самолет должен был остановиться (или выкатиться, если ПИДы не справились)
#     time.sleep(5)
#
#     # Сбрасываем всё обратно
#     fail_mgr.clear_all_failures()

In [39]:
# TODO
# class Telemetry:
#     def __init__(self, xpc: XPlaneConnectX):
#         self.xpc = xpc


class RunwayTracker:
    def __init__(self):
        self.R = 6371000.0  # Радиус Земли в метрах

    def _haversine_distance(self, lat1: float, lon1: float, lat2: float, lon2: float):
        phi1, phi2 = np.radians(lat1), np.radians(lat2)
        dphi = np.radians(lat2 - lat1)
        dlambda = np.radians(lon2 - lon1)
        a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
        return self.R * 2 * np.atan2(np.sqrt(a), np.sqrt(1 - a))

    @staticmethod
    def _bearing(lat1: float, lon1: float, lat2: float, lon2: float):
        phi1, phi2 = np.radians(lat1), np.radians(lat2)
        dlambda = np.radians(lon2 - lon1)
        y = np.sin(dlambda) * np.cos(phi2)
        x = np.cos(phi1) * np.sin(phi2) - np.sin(phi1) * np.cos(phi2) * np.cos(dlambda)
        return np.atan2(y, x)

    def get_cross_track_error(self, lat_ac: float, lon_ac: float):
        """Возвращает отклонение от осевой линии в метрах. >0 - правее оси, <0 - левее."""
        d_ac = self._haversine_distance(RWY_START_LAT, RWY_START_LON, lat_ac, lon_ac) / self.R
        theta_ac = self._bearing(RWY_START_LAT, RWY_START_LON, lat_ac, lon_ac)
        theta_rwy = self._bearing(RWY_START_LAT, RWY_START_LON, RWY_END_LAT, RWY_END_LON)

        xte = np.asin(np.sin(d_ac) * np.sin(theta_ac - theta_rwy)) * self.R
        return xte


class ReferenceTrajectory:
    """Генератор эталонной кривой скорости (равнозамедленное движение)."""

    def __init__(self, v_start_kts: float, v_target_kts: float, braking_distance_m: float):
        self.v_start_ms = v_start_kts * Converts.KTS_TO_MS
        self.v_target_ms = v_target_kts * Converts.KTS_TO_MS
        self.distance = braking_distance_m

        # Вычисление требуемого постоянного отрицательного ускорения (модуль)
        # Формула: a = (v_start^2 - v_tgt^2) / (2 * S)
        self.a_req = (self.v_start_ms ** 2 - self.v_target_ms ** 2) / (2.0 * self.distance)

    def get_reference_speed(self, current_distance_m: float) -> float:
        """Возвращает идеальную скорость (м/с) для текущей точки пути."""
        if current_distance_m >= self.distance:
            return self.v_target_ms

        # v(s) = sqrt(v_0^2 - 2 * a * s)
        val_under_sqrt = self.v_start_ms ** 2 - 2.0 * self.a_req * current_distance_m

        if val_under_sqrt <= 0:
            return self.v_target_ms

        return np.sqrt(val_under_sqrt)

In [40]:
class PIDController:
    """Пропорционально-интегрально-дифференциальный регулятор."""

    def __init__(self, kp: float, ki: float, kd: float, min_out: float = 0.0, max_out: float = 1.0,
                 anti_windup: float = 50.0, name: str = ""):
        self.kp = kp
        self.ki = ki
        self.kd = kd

        self.min_out = min_out
        self.max_out = max_out

        self.integral = 0.0
        self.prev_error = 0.0

        self.anti_windup = anti_windup
        self.name = name

    def compute(self, error: float, dt: float):
        if dt <= 0.0:
            return 0.0

        self.integral += error * dt
        # Anti-windup (защита от накопления интеграла)
        self.integral = max(-self.anti_windup, min(self.anti_windup, self.integral))

        derivative = (error - self.prev_error) / dt
        self.prev_error = error

        output = (self.kp * error) + (self.ki * self.integral) + (self.kd * derivative)

        return max(self.min_out, min(self.max_out, output))

    def reset(self):
        """Сброс внутренних состояний (используется при выключении системы)."""
        self.integral = 0.0
        self.prev_error = 0.0

In [41]:
class SystemBase:
    def __init__(self, xpc: XPlaneConnectX, failure_manager: FailureManager):
        self.xpc = xpc
        self.break_control = False
        self.failures = failure_manager

    def _control(self, dt: float):
        pass

    def control_step(self, dt: float):
        if self.break_control:
            return

        self._control(dt)

    def control_exception(self):
        pass


class RunwayCenteringSystem(SystemBase):
    def __init__(self, xpc: XPlaneConnectX, pid: PIDController, tracker: RunwayTracker,
                 failure_manager: FailureManager):
        super().__init__(xpc, failure_manager)
        self.pid = pid
        self.tracker = tracker

    def _control(self, dt: float):
        lat = self.xpc.current_dref_values["sim/flightmodel/position/latitude"]['value']
        lon = self.xpc.current_dref_values["sim/flightmodel/position/longitude"]['value']

        if lat is None or lon is None:
            cprint(f"[RunwayCenteringSystem] Error: drefs values is None", "red")
            return

        # Вычисление ошибки
        xte = self.tracker.get_cross_track_error(lat, lon)

        # Вычисление управляющего сигнала
        # Если мы правее оси (xte > 0), нам нужно повернуть влево (руль < 0).
        # Поэтому берем ошибку с обратным знаком.
        rudder_cmd = self.pid.compute(-xte, dt)
        # Добавляем обработку отказа управления носовой стойкой (NWS)
        rudder_cmd *= self.failures.state.steering_eff

        # Отправка команды в X-Plane (DataRef: sim/joystick/yoke_heading_ratio)
        self.xpc.sendDREF("sim/joystick/yoke_heading_ratio", rudder_cmd)

        print(f"[RunwayCenteringSystem] XTE: {xte:+.2f} м | Руль: {rudder_cmd:+.3f} | dt: {dt:.3f}с")

    def control_exception(self):
        print("\n[RunwayCenteringSystem] Остановка алгоритма. Возврат управления педалями в ноль.")
        self.xpc.sendDREF("sim/joystick/yoke_heading_ratio", 0.0)


class MultiChannelAutoBrake(SystemBase):
    def __init__(self, xpc: XPlaneConnectX, pid_brake_l: PIDController, pid_brake_r: PIDController,
                 pid_rev_l: PIDController, pid_rev_r: PIDController, trajectory: ReferenceTrajectory,
                 failure_manager: FailureManager):
        super().__init__(xpc, failure_manager)
        self.trajectory = trajectory

        self.pid_brake_l = pid_brake_l
        self.pid_brake_r = pid_brake_r
        self.pid_rev_l = pid_rev_l
        self.pid_rev_r = pid_rev_r

        self.traveled_distance_m = 0.0

        print("[MultiChannelAutoBrake] ИСМПУ: Запуск 4-канального торможения. Ожидание...")

    def _control(self, dt: float):
        current_speed_ms = self.xpc.current_dref_values["sim/flightmodel/position/groundspeed"]['value']

        if current_speed_ms is None:
            cprint(f"[MultiChannelAutoBrake] Error: drefs values is None", "red")
            return

        current_speed_kts = current_speed_ms * Converts.MS_TO_KTS

        self.traveled_distance_m += current_speed_ms * dt
        ref_speed_ms = self.trajectory.get_reference_speed(self.traveled_distance_m)

        # Глобальная ошибка по скорости. >0 означает, что мы едем слишком быстро
        error = current_speed_ms - ref_speed_ms

        # 1. Расчет тормозов (Hydraulic Brakes)
        cmd_brake_l = self.pid_brake_l.compute(error, dt)
        cmd_brake_r = self.pid_brake_r.compute(error, dt)

        # 2. Расчет реверса (Thrust Reversers) с учетом эксплуатационного лимита
        cmd_rev_l = 0.0
        cmd_rev_r = 0.0

        if current_speed_kts > 60.0:
            # Скорость безопасна для реверса
            cmd_rev_l = -self.pid_rev_l.compute(error, dt)  # Инверсия знака для X-Plane
            cmd_rev_r = -self.pid_rev_r.compute(error, dt)
        else:
            # Скорость ниже 60 узлов - принудительное отключение реверса.
            # Сбрасываем интеграторы, чтобы PID не копил ошибку, пока отключен.
            self.pid_rev_l.reset()
            self.pid_rev_r.reset()

        # Добавляет обработку некорректной конфигурации шасси (имитация неполного контакта стойки с ВПП)
        cmd_brake_l *= self.failures.state.brake_left_eff
        cmd_brake_r *= self.failures.state.brake_right_eff

        # Добавляет обработку нарушения тяги / реверса двигателей
        cmd_rev_l *= self.failures.state.reverse_left_eff
        cmd_rev_l *= self.failures.state.thrust_left_eff

        cmd_rev_r *= self.failures.state.reverse_right_eff
        cmd_rev_r *= self.failures.state.thrust_right_eff

        # 3. Отправка команд в X-Plane по независимым каналам
        self.xpc.sendDREF("sim/cockpit2/controls/left_brake_ratio", cmd_brake_l)
        self.xpc.sendDREF("sim/cockpit2/controls/right_brake_ratio", cmd_brake_r)

        # Индексация массивов: [0] - левый двигатель, [1] - правый двигатель
        self.xpc.sendDREF("sim/cockpit2/engine/actuators/throttle_ratio[0]", cmd_rev_l)
        self.xpc.sendDREF("sim/cockpit2/engine/actuators/throttle_ratio[1]", cmd_rev_r)

        print(f"[MultiChannelAutoBrake] Dist: {self.traveled_distance_m:4.0f}m | V_cur: {current_speed_kts:3.0f} | "
              f"Brk_L: {cmd_brake_l:.2f} | Brk_R: {cmd_brake_r:.2f} | "
              f"Rev_L: {cmd_rev_l:.2f} | Rev_R: {cmd_rev_r:.2f}")

        self.last_time = current_time

        if current_speed_kts * Converts.KTS_TO_MS <= self.trajectory.v_target_ms:
            print("[MultiChannelAutoBrake] Посадочная дистанция пройдена. Скорость руления достигнута.")
            self.xpc.sendDREF("sim/cockpit2/controls/left_brake_ratio", 0.1)
            self.xpc.sendDREF("sim/cockpit2/controls/right_brake_ratio", 0.1)
            self.break_control = True
            return

    def control_exception(self):
        print("\n[MultiChannelAutoBrake] Остановка. Сброс всех управляющих поверхностей.")
        self.xpc.sendDREF("sim/cockpit2/controls/left_brake_ratio", 0.0)
        self.xpc.sendDREF("sim/cockpit2/controls/right_brake_ratio", 0.0)
        self.xpc.sendDREF("sim/cockpit2/engine/actuators/throttle_ratio[0]", 0.0)
        self.xpc.sendDREF("sim/cockpit2/engine/actuators/throttle_ratio[1]", 0.0)


In [42]:
# # Коэффициенты подобраны эмпирически. Для тяжелых ЛА (B737) требуется меньше P и больше D.
# # Так как руль направления имеет обратную логику (чтобы повернуть влево (xte < 0), нужно дать отрицательный ввод),
# # знак вывода PID может потребовать инверсии в зависимости от настроек симулятора.
# runway_center_pid = PIDController(kp=0.0006, ki=0.0004, kd=0.045, min_out=-1, max_out=1)
# # runway_center_pid = PIDController(kp=0.0002, ki=0.0007, kd=0.05, min_out=-50, max_out=50)
#
# # Коэффициенты PID подобраны для тяжелого ЛА (масса ~200 тонн)
# # Требуется калибровка под конкретную динамику A330 в X-Plane
# velocity_pid = PIDController(kp=0.15, ki=0.01, kd=0.05, min_out=-50, max_out=50)
#
# # Инициализация 4 независимых контроллеров
# # Коэффициенты для реверса сделаны меньше, так как тяга двигателей
# # имеет огромную инерцию по сравнению с гидравликой тормозов.
# pid_brake_l = PIDController(kp=0.1, ki=0.01, kd=0.05, min_out=-50, max_out=50, name="Brake_L")
# pid_brake_r = PIDController(kp=0.1, ki=0.01, kd=0.05, min_out=-50, max_out=50, name="Brake_R")
#
# # Реверс в X-Plane управляется отрицательным значением throttle_ratio [-1.0, 0.0]
# # Мы настраиваем PID на выдачу от 0 до 1, а инвертировать знак будем перед отправкой.
# # pid_rev_l = PIDController(kp=0.15, ki=0.02, kd=0.0, min_out=0.0, max_out=50, name="Rev_L")
# # pid_rev_r = PIDController(kp=0.15, ki=0.02, kd=0.0, min_out=0.0, max_out=50, name="Rev_R")
# pid_rev_l = PIDController(kp=0.03, ki=0.002, kd=0.01, min_out=0.0, max_out=50, name="Rev_L")
# pid_rev_r = PIDController(kp=0.03, ki=0.002, kd=0.01, min_out=0.0, max_out=50, name="Rev_R")

In [44]:
# Инициализация асинхронной подписки на необходимые параметры с частотой 20 Гц
subscribed_drefs = [
    ("sim/flightmodel/position/latitude", 20),
    ("sim/flightmodel/position/longitude", 20),
    ("sim/flightmodel/position/groundspeed", 20)
]
print("Настройка подписки на телеметрию X-Plane...")
xpc.subscribeDREFs(subscribed_drefs, timeout=10.)
time.sleep(2)


tracker = RunwayTracker()
trajectory = ReferenceTrajectory(
    v_start_kts=INITIAL_SPEED_KTS,
    v_target_kts=TARGET_SPEED_KTS,
    braking_distance_m=2000.0
)

failure_manager = FailureManager()


# default
# runway_center_pid = PIDController(kp=0.0006, ki=0.0004, kd=0.045, min_out=-1, max_out=1)
# pid_brake_l = PIDController(kp=0.1, ki=0.01, kd=0.05, min_out=-50, max_out=50, name="Brake_L")
# pid_brake_r = PIDController(kp=0.1, ki=0.01, kd=0.05, min_out=-50, max_out=50, name="Brake_R")
# pid_rev_l = PIDController(kp=0.03, ki=0.002, kd=0.01, min_out=0.0, max_out=50, name="Rev_L")
# pid_rev_r = PIDController(kp=0.03, ki=0.002, kd=0.01, min_out=0.0, max_out=50, name="Rev_R")

# left reverse fail
failure_manager.activate(FailureMode.REVERSE_LEFT_FAIL)
runway_center_pid = PIDController(kp=0.0006, ki=0.0004, kd=0.045, min_out=-1, max_out=1)
pid_brake_l = PIDController(kp=0.1, ki=0.01, kd=0.05, min_out=-50, max_out=50, name="Brake_L")
pid_brake_r = PIDController(kp=0.1, ki=0.01, kd=0.05, min_out=-50, max_out=50, name="Brake_R")
pid_rev_l = PIDController(kp=0.03, ki=0.002, kd=0.01, min_out=0.0, max_out=50, name="Rev_L")
pid_rev_r = PIDController(kp=0.03, ki=0.002, kd=0.01, min_out=0.0, max_out=50, name="Rev_R")


centering = RunwayCenteringSystem(xpc, runway_center_pid, tracker, failure_manager)
multi_autobrake = MultiChannelAutoBrake(xpc, pid_brake_l, pid_brake_r, pid_rev_l, pid_rev_r, trajectory,
                                        failure_manager)

time.sleep(2)
setup_touchdown_uuee_24c(xpc, **A330_SETUP)

print("Запуск системы удержания оси ВПП...")
last_time = time.time()

try:
    while True:
        current_time = time.time()
        dt = current_time - last_time

        if dt >= 0.05:  # Частота контура 20 Гц

            centering.control_step(dt)
            multi_autobrake.control_step(dt)

            last_time = current_time

        time.sleep(0.01)  # Снижение нагрузки на CPU

except KeyboardInterrupt:
    centering.control_exception()
    multi_autobrake.control_exception()


Настройка подписки на телеметрию X-Plane...
[MultiChannelAutoBrake] ИСМПУ: Запуск 4-канального торможения. Ожидание...
Запуск системы удержания оси ВПП...
[RunwayCenteringSystem] XTE: +0.22 м | Руль: -0.192 | dt: 0.052с
[MultiChannelAutoBrake] Dist:    4m | V_cur: 160 | Brk_L: 0.06 | Brk_R: 0.06 | Rev_L: -0.00 | Rev_R: -0.01
[RunwayCenteringSystem] XTE: +0.17 м | Руль: +0.044 | dt: 0.051с
[MultiChannelAutoBrake] Dist:    8m | V_cur: 160 | Brk_L: -0.01 | Brk_R: -0.01 | Rev_L: -0.00 | Rev_R: -0.00
[RunwayCenteringSystem] XTE: +0.17 м | Руль: -0.000 | dt: 0.052с
[MultiChannelAutoBrake] Dist:   13m | V_cur: 160 | Brk_L: 0.10 | Brk_R: 0.10 | Rev_L: -0.00 | Rev_R: -0.02
[RunwayCenteringSystem] XTE: +0.30 м | Руль: -0.115 | dt: 0.052с
[MultiChannelAutoBrake] Dist:   17m | V_cur: 160 | Brk_L: -0.02 | Brk_R: -0.02 | Rev_L: -0.00 | Rev_R: -0.00
[RunwayCenteringSystem] XTE: +0.11 м | Руль: +0.165 | dt: 0.052с
[MultiChannelAutoBrake] Dist:   21m | V_cur: 159 | Brk_L: -0.07 | Brk_R: -0.07 | Rev_L: 